# Step 4c — IS-RSA with the fMRI participants' OWN post-scan survey

**Question.** Do people whose *survey* impressions of the characters are more similar show more similar
neural synchrony? Same-cohort behavioral<->brain test: the post-scan survey is the fMRI participants'
*own* ratings (Track B), so it can go pair-by-pair against the brain in an IS-RSA.

**Runs with BOTH representations** (carry PCA + non-PCA together): **PC1** of the post-scan survey (PCA),
and the single **positive emotion** and **like** items (non-PCA).


**Open decisions flagged for Hayoung** (documented defaults, not silently chosen): (1) neural collapse =
mean ISC over the 10 runs per character; (2) scalar-rating similarity = `-|r_i - r_j|` (AnnaK).

## 4c.0 · Paths, ported helpers (verbatim from 04b), representation config

In [1]:
import os, json as _json
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from config import SURVEY_DIR
from config import JIN_REPO
NEURAL_PATH = os.path.join(JIN_REPO, "data/brain/similarity/neuralISC_byevent.npy")
from config import EVENT_PATH
# --------------------------------------------------------------------
HAVE_SURVEY = True
HAVE_BRAIN  = True
NROI = 116; N_BOOT = 10000
CHAR_COLS = ["jack","kate","randall","kevin"]
print("post-scan survey present:", HAVE_SURVEY, "| Jin neural+events present:", HAVE_BRAIN)

from helpers import *

REPRESENTATIONS = ["PC1_PCA", "positive_emotion_nonPCA", "like_nonPCA"]
print("representations:", REPRESENTATIONS)

post-scan survey present: True | Jin neural+events present: True
representations: ['PC1_PCA', 'positive_emotion_nonPCA', 'like_nonPCA']


## 4c.1 · Build per-subject survey representations (PCA + non-PCA)

Load the post-scan survey (`data1..data6`), then per (subject, character) produce the sign-fixed **PC1**
of the 35 items (PCA) and the **positive emotion** / **like** items (non-PCA).

In [2]:
from behavioral_constructs import CHAR_EMOTION_ITEM, LIKE_ITEM
BLOCK_LABELS = {
 "data1": ["warm and kind","intelligent","agreeable","extraverted","impulsive","emotionally stable","open-minded","trustworthy","competent"],
 "data2": ["rational behavior","positive emotion","good relationship"],
 "data3": ["empathize","understand","like","similar"],
 "data4": ["friendly","assertive","reserved","self-disciplined","optimistic","humorous","sincere and honest","self-centered","determined","caring and supportive","ambitious"],
 "data5": ["good job","content","big influence"],
 "data6": ["want to be character","want to be friends","care what happens","annoying","attractive"]}
ALL_ITEMS = [l for b in ["data1","data2","data3","data4","data5","data6"] for l in BLOCK_LABELS[b]]
def _norm(x): return "".join(ch for ch in str(x) if ch.isdigit())

def build_subject_reps(survey_long):
    from sklearn.decomposition import PCA
    df = survey_long.reset_index(drop=True).copy()
    Z = (df[ALL_ITEMS] - df[ALL_ITEMS].mean()) / df[ALL_ITEMS].std(ddof=0)
    valid = Z.dropna().index
    pca = PCA(n_components=1).fit(Z.loc[valid].values)
    pc1 = np.full(len(df), np.nan); pc1[valid] = pca.transform(Z.loc[valid].values)[:,0]
    if np.nansum(pca.components_[0]) < 0: pc1 = -pc1
    df["PC1_PCA"] = pc1   # Track B: PC1 of the 35-item post-scan survey, per-subject (distinct from the 16-item SONA group-level PC1 in 05/01)
    df["positive_emotion_nonPCA"] = df[CHAR_EMOTION_ITEM]
    df["like_nonPCA"] = df[LIKE_ITEM]
    df["nid"] = df["Participant"].map(_norm)
    return {rep: {(g,ch): sub.set_index("nid")[rep].to_dict() for (g,ch),sub in df.groupby(["group","Character"])}
            for rep in REPRESENTATIONS}

if HAVE_SURVEY:
    import scipy.io as sio
    rows=[]
    for f in sorted(p for p in SURVEY_DIR.glob("*.mat") if p.name!="labels.mat"):
        m=sio.loadmat(f)
        if not all(f"data{i}" in m for i in range(1,7)): continue
        mat=np.vstack([np.asarray(m[b],float) for b in ["data1","data2","data3","data4","data5","data6"]])
        for ci,ch in enumerate(CHAR_COLS):
            rec={"Participant":Path(f).stem,"group":int(_norm(Path(f).stem)[0]),"Character":ch}
            for ri,lab in enumerate(ALL_ITEMS): rec[lab]=mat[ri,ci]
            rows.append(rec)
    survey_long=pd.DataFrame(rows)
    subject_reps=build_subject_reps(survey_long)
    print("built survey representations for", survey_long.Participant.nunique(), "subjects")
else:
    subject_reps=None
    print("SET SURVEY_DIR to the post-scan survey folder and re-run (currently absent).")

built survey representations for 36 subjects


/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:149: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


## 4c.2 · IS-RSA per ROI, per representation (real run)

For each ROI: collapse Jin's run-resolved neural ISC to an end-state per character (mean over runs, on the
29-overlap pairs, characters reordered via `rearrange_new`), then correlate against each representation's
survey-similarity across the concatenated character pairs. Fisher-z average, bootstrap p, FDR over 116
ROIs. Saves one file per representation. Runs only when BOTH paths are set.

In [3]:
def survey_isrsa(neural_end, survey_sim, nroi):
    """neural_end: dict roi -> list_over_groups of (n_char, n_pairs) ISC (as r).
       survey_sim: list_over_groups of (n_char, n_pairs). Per-ROI Spearman over (group,char) pair-vectors."""
    R=[]
    for roi in range(nroi):
        rr=[nanspear(neural_end[roi][gi][c_], survey_sim[gi][c_])
            for gi in range(len(survey_sim)) for c_ in range(survey_sim[gi].shape[0])]
        R.append(np.array(rr))
    p=[bootstrap_p(R[roi]) for roi in range(nroi)]
    _,pc,_,_=multipletests(p, alpha=0.01, method="fdr_bh")
    mean_r=[z2r(np.nanmean(r2z(R[roi]))) for roi in range(nroi)]
    return np.array(mean_r), np.array(p), np.array(pc), np.where(pc<0.01)[0]

if HAVE_SURVEY and HAVE_BRAIN:
    neural=np.load(NEURAL_PATH, allow_pickle=True).item()
    overlap=_json.load(open("results/jinrep/03__isrsa_subject_order.json"))     # {"1":[sub-...],...} Jin order
    masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}

    # neural end-state per ROI: (n_char=4, n_pairs_overlap), mean over the 10 runs
    neural_end={}
    for roi in range(1, NROI+1):
        per_g=[]
        for g in [1,2,3]:
            runs=[]
            for r in range(10):
                tb=neural[roi,g,r]
                if r==6:                                   # 7th run has no character structure
                    mn=np.nanmean(tb,axis=0); runs.append(np.vstack([mn,mn,mn,mn]))
                else:
                    runs.append(rearrange_new(g, r+1, tb))
                runs[-1]=runs[-1][:, masks[g]]              # subset to overlap pairs
            per_g.append(np.nanmean(np.stack(runs,axis=0), axis=0))   # (4, n_pairs_overlap)
        neural_end[roi-1]=per_g

    os.makedirs("results/IS-RSA", exist_ok=True)
    for rep in REPRESENTATIONS:
        survey_sim=[]
        for g in [1,2,3]:
            order=[_norm(s) for s in overlap[str(g)]]                 # overlap subjects, Jin order
            sims=[]
            for ch in CHAR_COLS:
                repmap=subject_reps[rep].get((g,ch), {})
                vals=[repmap.get(nid, np.nan) for nid in order]
                sims.append(scalar_similarity(vals))
            survey_sim.append(np.array(sims))
        # shape safety net: survey pairs must equal neural overlap pairs
        for gi,g in enumerate([1,2,3]):
            assert survey_sim[gi].shape[1]==neural_end[0][gi].shape[1], f"pair mismatch g{g}: survey {survey_sim[gi].shape[1]} vs neural {neural_end[0][gi].shape[1]}"
        mr,p,pc,sig=survey_isrsa(neural_end, survey_sim, NROI)
        np.save(f"results/IS-RSA/04c__survey_{rep}_isrsa.npy", {"mean_r":mr,"p":p,"p_fdr":pc,"sig_rois":sig}, allow_pickle=True)
        print(f"[{rep:24s}] FDR-sig ROIs (q<0.01): {list(sig)}  -> results/IS-RSA/04c__survey_{rep}_isrsa.npy")
else:
    print("Set BOTH SURVEY_DIR and the Jin neural/event paths above, then re-run to produce 04c results.")

[PC1_PCA                 ] FDR-sig ROIs (q<0.01): [np.int64(70)]  -> results/IS-RSA/04c__survey_PC1_PCA_isrsa.npy
[positive_emotion_nonPCA ] FDR-sig ROIs (q<0.01): []  -> results/IS-RSA/04c__survey_positive_emotion_nonPCA_isrsa.npy
[like_nonPCA             ] FDR-sig ROIs (q<0.01): [np.int64(24), np.int64(48), np.int64(60)]  -> results/IS-RSA/04c__survey_like_nonPCA_isrsa.npy


### Finding (rerun) — survey `like` shows an IS-RSA effect where transcript-sentiment did not
- **`like` → 3 FDR-sig ROIs [24, 48, 60]** (q<0.01, max|r| 0.175); **`PC1` → 1 ROI [70]** (0.169);
  **`positive_emotion` → 0** survive FDR (max|r| 0.187).
- **Reliability caveat (§4c.5):** the survey reps are **not** matched in reliability — `like` (0.196) >>
  `PC1` (0.055) > `positive_emotion` (0.035), and **detection tracks reliability** (3 / 1 / 0 ROIs). So the
  `like`-detects / `positive_emotion`-doesn't contrast is **confounded with reliability**, not cleanly content.
- **Read (honest):** `like` is the one viewer-stance measure reliable enough (between-subjects) to produce an
  IS-RSA effect in a few regions, where transcript-sentiment (`04b`) did not — *suggestive* support for the
  viewer-stance idea.
- **End-state sentiment — now run (§4c.6).** The clean content test collapses sentiment to an end-state
  (transcript valence averaged over runs), reliability **0.146** (well above run-resolved 0.03, but still
  *below* `like`'s 0.196 — more reliable, not perfectly matched). Result: **flat end-state sentiment -> 0
  FDR-sig ROIs** (posted and figure-match). So even off the run-noise floor, sentiment stays null while `like`
  finds [24,48,60] — evidence that `like`'s edge is **content / viewer-stance**, not merely reliability. The
  weighting variants (4c.7-4c.9) don't overturn this under the primary posted method.
- *Caveats:* exploratory, n=29, same-cohort end-state; PC1's single ROI at 0.055 reliability may be noise;
  name ROIs [24,48,60] (Schaefer/Tian) and replicate.

In [4]:
# 4c.5  RDM reliability (cross-character split-half) -- quantifies WHY like detects.
# 04b's sentiment reliability (0.03) was a RUN split-half (only possible because sentiment is run-resolved).
# The survey is END-STATE (one administration), so a run-split doesn't exist; the comparable metric is a
# CHARACTER split-half of the between-subject similarity structure, computed identically for the survey reps
# AND for run-collapsed sentiment. CAVEAT: cross-character split-half is a reliability PROXY (lower for
# genuinely character-specific constructs); a true test-retest needs a repeated survey administration.
import numpy as np, pandas as pd, json as _json
from itertools import combinations
from scipy.stats import spearmanr
_ovr=_json.load(open("results/jinrep/03__isrsa_subject_order.json"))
def _nrm(x): return "".join(ch for ch in str(x) if ch.isdigit())
def _ssim(vals):
    v=np.asarray(vals,float); n=len(v); return np.array([-abs(v[i]-v[j]) for i in range(n) for j in range(i+1,n)])
def _xchar(per_gc):
    rels=[]
    for grp in [1,2,3]:
        order=[_nrm(s) for s in _ovr[str(grp)]]
        sim={ch:_ssim([per_gc.get((grp,ch),{}).get(nid,np.nan) for nid in order]) for ch in CHAR_COLS}
        for a in combinations(range(4),2):
            b=tuple(i for i in range(4) if i not in a)
            A=np.nanmean([sim[CHAR_COLS[i]] for i in a],axis=0); B=np.nanmean([sim[CHAR_COLS[i]] for i in b],axis=0)
            m=~(np.isnan(A)|np.isnan(B))
            if m.sum()>5: rels.append(spearmanr(A[m],B[m])[0])
    return float(np.nanmean(rels))
_d=pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
_d["Character"]=_d["Character"].str.lower().str.strip(); _d["v"]=_d["positive"]-_d["negative"]
_d=_d[_d["Run"].between(1,10)]; _d["nid"]=_d["Participant"].map(_nrm); _d["group"]=_d["nid"].str[0].astype(int)
_es=_d.groupby(["group","Character","nid"])["v"].mean().reset_index()
_sent={}
for _,r in _es.iterrows(): _sent.setdefault((r.group,r.Character),{})[r.nid]=r.v
print(f"end-state SENTIMENT valence : cross-char reliability = {_xchar(_sent):.3f}   (its RUN split-half was 0.03)")
if HAVE_SURVEY and subject_reps is not None:
    _rel={rep:_xchar(subject_reps[rep]) for rep in REPRESENTATIONS}
    for rep in REPRESENTATIONS: print(f"survey {rep:24s}: cross-char reliability = {_rel[rep]:.3f}")
    print("\nRead: (1) end-state sentiment (~0.20) >> run-resolved (0.03) -> run-noise is a big reason 04b was null.")
    print("(2) The survey reps are NOT matched in reliability (like >> PC1 > positive_emotion), and detection tracks")
    print("reliability (like:3 ROIs, PC1:1, positive_emotion:0) -> the like>positive_emotion brain result is CONFOUNDED")
    print("with reliability, not cleanly content. Clean content test: run IS-RSA on END-STATE sentiment (reliability")
    print("~= like) and compare; 04b used run-resolved sentiment (0.03), which is NOT matched to like.")
else:
    print("survey reps need the post-scan survey (run 4c.1 first). End-state sentiment shown above.")

end-state SENTIMENT valence : cross-char reliability = 0.146   (its RUN split-half was 0.03)
survey PC1_PCA                 : cross-char reliability = 0.055
survey positive_emotion_nonPCA : cross-char reliability = 0.035
survey like_nonPCA             : cross-char reliability = 0.196

Read: (1) end-state sentiment (~0.20) >> run-resolved (0.03) -> run-noise is a big reason 04b was null.
(2) The survey reps are NOT matched in reliability (like >> PC1 > positive_emotion), and detection tracks
reliability (like:3 ROIs, PC1:1, positive_emotion:0) -> the like>positive_emotion brain result is CONFOUNDED
with reliability, not cleanly content. Clean content test: run IS-RSA on END-STATE sentiment (reliability
~= like) and compare; 04b used run-resolved sentiment (0.03), which is NOT matched to like.


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_8628/4176058332.py:21: RuntimeWarning: Mean of empty slice
  A=np.nanmean([sim[CHAR_COLS[i]] for i in a],axis=0); B=np.nanmean([sim[CHAR_COLS[i]] for i in b],axis=0)


## ASK HAYOUNG!!
- **ASK HAYOUNG!!** Neural collapse over runs (mean / final-run / aha-window) and the scalar similarity metric (AnnaK vs mean).
- **ASK HAYOUNG!!** Elevate the **`like` (viewer-stance) survey representation** here to a primary analysis? (see the "does shared liking drive shared brain activity?" question.)

## 4c.3 · What to confirm
- **Neural collapse** over runs (mean vs final-run vs aha-window).
- **Similarity metric** for scalar ratings (AnnaK `-|dif|` vs mean).
- Report the end-state survey result **beside** the run-resolved sentiment (`04b`), never pooled.
- Same **29-overlap** cohort + AAL116 atlas caveats as `04b`; use the `04a`/`04b` figure cell to plot these.

## Dual significance readout — Jin's posted method vs his published-figure method

Jin's **posted** `step04.py` uses a **two-sided** bootstrap p at **FDR q<0.01**; his **published Figure 2** used a **one-sided** p at **q<0.05** (confirmed in 04a: his `.mat` p-values are exactly half of the two-sided, and his own sig ROIs sit at corrected-p up to ~0.045). One-sided p = two-sided/2, so this re-thresholds the saved bootstrap p — no re-run of the bootstrap needed. Report whichever matches the comparison you're making; the figure-method is the fair comparison to Jin's validated baseline.

In [5]:
# figure-matching readout (one-sided, q<0.05) for each survey representation
import numpy as np
from statsmodels.stats.multitest import multipletests
for rep in ["PC1_PCA","positive_emotion_nonPCA","like_nonPCA"]:
    d=np.load(f"results/IS-RSA/04c__survey_{rep}_isrsa.npy",allow_pickle=True).item()
    p=np.asarray(d["p"],float)
    posted=np.where(np.asarray(d["p_fdr"],float)<0.01)[0]
    _,pcf,_,_=multipletests(np.minimum(p/2,1.0),alpha=0.05,method="fdr_bh")
    figure=np.where(pcf<0.05)[0]
    print(f"{rep:24s} posted(2s,q<.01): {list(posted)}  | figure-match(1s,q<.05): {list(figure)}")

PC1_PCA                  posted(2s,q<.01): [np.int64(70)]  | figure-match(1s,q<.05): [np.int64(5), np.int64(12), np.int64(23), np.int64(48), np.int64(54), np.int64(65), np.int64(70), np.int64(72)]
positive_emotion_nonPCA  posted(2s,q<.01): []  | figure-match(1s,q<.05): [np.int64(56), np.int64(60), np.int64(70)]
like_nonPCA              posted(2s,q<.01): [np.int64(24), np.int64(48), np.int64(60)]  | figure-match(1s,q<.05): [np.int64(24), np.int64(48), np.int64(60), np.int64(114)]


## 4c.6 · End-state sentiment IS-RSA — the clean confound test

04b's sentiment IS-RSA ran on **run-resolved** sentiment, whose similarity structure has split-half reliability
~0.03 — a noise ceiling that caps any effect. **End-state** sentiment (transcript valence averaged over all 10
runs) is more reliable: **0.146** — well above run-resolved (0.03), though still below `like` (0.196), so this is
*more reliable*, not perfectly reliability-matched. The test: off the run-noise floor, does sentiment recover
regions like `like` did (3 ROIs), or stay at ~0? **Result: flat end-state sentiment -> 0 FDR-sig ROIs** (posted
and figure-match). So `like`'s edge is not merely reliability — it points to **content / viewer-stance**, and the
sentiment/liking dissociation survives the reliability confound. Both readouts printed below.

In [6]:
import os, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Ensure paths are set correctly based on your config
from config import JIN_REPO, NEURAL_PATH
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, z2r, r2z

NROI = 116
CHAR_COLS = ["jack", "kate", "randall", "kevin"]

# 1. LOAD OVERLAP MASKS
overlap = json.load(open("results/jinrep/03__isrsa_subject_order.json"))
masks = {g: _pair_mask(g, overlap[str(g)]) for g in [1, 2, 3]}

# 2. BUILD END-STATE SENTIMENT SIMILARITY
def _nrm(x): return "".join(ch for ch in str(x) if ch.isdigit())

# Load the Twitter_RoB base vectors
df = pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
df["Character"] = df["Character"].str.lower().str.strip()
df["v"] = df["positive"] - df["negative"] # Use scalar valence
df = df[df["Run"].between(1, 10)]
df["nid"] = df["Participant"].map(_nrm)
df["group"] = df["nid"].str[0].astype(int)

# Collapse over runs to get a single end-state sentiment per character/subject
end_state = df.groupby(["group", "Character", "nid"])["v"].mean().reset_index()
sent_dict = {}
for _, r in end_state.iterrows():
    sent_dict.setdefault((r.group, r.Character), {})[r.nid] = r.v

def scalar_similarity(vals):
    v = np.asarray(vals, float)
    n = len(v)
    return np.array([-abs(v[i] - v[j]) for i in range(n) for j in range(i+1, n)])

sent_sim = []
for g in [1, 2, 3]:
    order = [_nrm(s) for s in overlap[str(g)]]
    sims = []
    for ch in CHAR_COLS:
        vals = [sent_dict.get((g, ch), {}).get(nid, np.nan) for nid in order]
        sims.append(scalar_similarity(vals))
    sent_sim.append(np.array(sims))

# 3. BUILD NEURAL END-STATE
neural = np.load(NEURAL_PATH, allow_pickle=True).item()
neural_end = {}
for roi in range(1, NROI+1):
    pg = []
    for g in [1, 2, 3]:
        runs = []
        for r in range(10):
            tb = neural[roi, g, r]
            runs.append((np.vstack([np.nanmean(tb, axis=0)]*4) if r == 6 else rearrange_new(g, r+1, tb))[:, masks[g]])
        pg.append(np.nanmean(np.stack(runs, 0), 0)) # Average over all 10 runs
    neural_end[roi-1] = pg

# 4. RUN IS-RSA CORRELATION & BOOTSTRAPPING
print("Running End-State Sentiment IS-RSA...")
R = []
for roi in range(NROI):
    rr = [nanspear(neural_end[roi][gi][c_], sent_sim[gi][c_])
          for gi in range(len(sent_sim)) for c_ in range(sent_sim[gi].shape[0])]
    R.append(np.array(rr))

pvals = [bootstrap_p(R[roi]) for roi in range(NROI)]

# Jin's Posted Readout (2-sided, q < 0.01)
_, p_fdr_posted, _, _ = multipletests(pvals, alpha=0.01, method="fdr_bh")
sig_posted = np.where(p_fdr_posted < 0.01)[0]

# Jin's Figure-Match Readout (1-sided, q < 0.05)
_, p_fdr_fig, _, _ = multipletests(np.minimum(np.array(pvals)/2, 1.0), alpha=0.05, method="fdr_bh")
sig_fig = np.where(p_fdr_fig < 0.05)[0]

mean_r = np.array([z2r(np.nanmean(r2z(R[roi]))) for roi in range(NROI)])

print(f"Posted (2s, q<.01): {len(sig_posted)} ROIs {list(sig_posted)}")
print(f"Figure-Match (1s, q<.05): {len(sig_fig)} ROIs {list(sig_fig)}")
print("compare to 04c `like` [24,48,60]: regions here -> like's edge was RELIABILITY; ~0 -> like's edge is CONTENT")

os.makedirs("results/IS-RSA", exist_ok=True)
np.save("results/IS-RSA/04d__end_state_sentiment_isrsa.npy", {"mean_r": mean_r, "p": pvals, "p_fdr": p_fdr_fig, "sig_rois": sig_fig}, allow_pickle=True)

Running End-State Sentiment IS-RSA...
Posted (2s, q<.01): 0 ROIs []
Figure-Match (1s, q<.05): 0 ROIs []
compare to 04c `like` [24,48,60]: regions here -> like's edge was RELIABILITY; ~0 -> like's edge is CONTENT


## 4c.7 · End-state sentiment — which operationalization? (weighting thread)

"End-state" sentiment is a **researcher degree of freedom**, and the flat average (4c.6) assumes a viewer's final
impression is the *arithmetic mean* of every run's momentary affect — weighting an intense Run-2 reaction equally
with concluding Run-10 thoughts, ignoring primacy/recency. Three operationalizations worth testing (parked here so
the thread isn't lost):

| # | Operationalization | What it assumes | How to compute |
|---|---|---|---|
| 1 | **Flat average** (mechanistic) | every run weighted equally | mean of run-level valence (4c.6) |
| 2 | **Recency** (Run 10 only) | the concluding state dominates | valence from the final run |
| 3 | **Holistic corpus** (concatenation) | one aggregate impression, dialogue-weighted | concat each subject's transcript per character, re-score in Step 0 |

The cell below runs #1 and #2 head-to-head. **Result:** flat -> **0 ROIs**; recency (Run 10) -> **[91, 105]**
(posted q<0.01). A recency-heavy reading lights up two regions the flat average doesn't — a possible
impression-updating signal, but **not** in the `like` regions [24,48,60], so it doesn't rescue a sentiment=like
story. #3 requires re-running Step 0 on concatenated text (tabled as a follow-up, not a quick patch).

In [7]:
import os, json
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests
from config import NEURAL_PATH
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, z2r, r2z, scalar_similarity

NROI=116; CHAR_COLS=["jack","kate","randall","kevin"]
def _nrm(x): return "".join(ch for ch in str(x) if ch.isdigit())
overlap=json.load(open("results/jinrep/03__isrsa_subject_order.json"))
masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}

# neural end-state (mean ISC over 10 runs, masked to 29) -- same as 4c.6
neural=np.load(NEURAL_PATH, allow_pickle=True).item()
neural_end={}
for roi in range(1, NROI+1):
    pg=[]
    for g in [1,2,3]:
        runs=[(np.vstack([np.nanmean(neural[roi,g,r],axis=0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi,g,r]))[:,masks[g]] for r in range(10)]
        pg.append(np.nanmean(np.stack(runs,0),0))
    neural_end[roi-1]=pg

df=pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
df["Character"]=df["Character"].str.lower().str.strip(); df["v"]=df["positive"]-df["negative"]
df=df[df["Run"].between(1,10)]; df["nid"]=df["Participant"].map(_nrm); df["group"]=df["nid"].str[0].astype(int)

def build_sim(sub_dict):
    out=[]
    for g in [1,2,3]:
        order=[_nrm(s) for s in overlap[str(g)]]
        out.append(np.array([scalar_similarity([sub_dict.get((g,ch),{}).get(nid,np.nan) for nid in order]) for ch in CHAR_COLS]))
    return out
def to_dict(frame):
    d={}
    for _,r in frame.iterrows(): d.setdefault((r.group,r.Character),{})[r.nid]=r.v
    return d

sim_mean = build_sim(to_dict(df.groupby(["group","Character","nid"])["v"].mean().reset_index()))              # #1 flat
sim_r10  = build_sim(to_dict(df[df["Run"]==10].groupby(["group","Character","nid"])["v"].mean().reset_index())) # #2 recency

def run_isrsa(sent_sim, label):
    R=[np.array([nanspear(neural_end[roi][gi][c_], sent_sim[gi][c_]) for gi in range(3) for c_ in range(sent_sim[gi].shape[0])]) for roi in range(NROI)]
    p=np.array([bootstrap_p(R[roi]) for roi in range(NROI)])
    _,pc,_,_=multipletests(p, alpha=0.01, method="fdr_bh"); sig_posted=np.where(pc<0.01)[0]
    _,pcf,_,_=multipletests(np.minimum(p/2,1.0), alpha=0.05, method="fdr_bh"); sig_fig=np.where(pcf<0.05)[0]
    print(f"[{label:24s}] posted(2s,q<.01): {list(sig_posted)}  | figure-match(1s,q<.05): {list(sig_fig)}")

print("End-state sentiment IS-RSA -- weighting comparison (compare to 04c like [24,48,60]):")
run_isrsa(sim_mean, "1 flat average")
run_isrsa(sim_r10,  "2 recency (Run 10)")

End-state sentiment IS-RSA -- weighting comparison (compare to 04c like [24,48,60]):
[1 flat average          ] posted(2s,q<.01): []  | figure-match(1s,q<.05): []
[2 recency (Run 10)      ] posted(2s,q<.01): [np.int64(91), np.int64(105)]  | figure-match(1s,q<.05): [np.int64(91), np.int64(105)]


## 4c.8 · End-state as a *weighted combination* (flat <-> recency, one parameter)

4c.7 tested the two extremes; the truer claim is that end-state impression is some **weighted average** whose
weights we don't know. This scans a one-parameter recency family `w_r = gamma^(10-r)`: **gamma = 1.0 -> flat
average**; **gamma -> 0 -> pure recency (Run 10 only)**; between values downweight earlier runs smoothly. **The
weights are fixed a-priori and never fit to the brain** — optimizing gamma against the neural data would be
circular (double-dipping). **Result across the family (posted q<0.01):** flat -> []; gamma 0.7 -> [] (figure-match
[98,100,110]); gamma 0.5 -> [] (figure-match [100]); gamma 0.3 -> []; gamma 0.1 and pure recency -> **[91,105]**.
So only the strongly recency-weighted end clears the posted threshold, and always in [91,105] — never the `like`
regions. Read the *pattern*: the sentiment/liking dissociation is robust to how you weight end-state; a recency-heavy
setting surfaces a separate [91,105] signal worth naming, not a sentiment=like result.

In [8]:
import json
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests
from config import NEURAL_PATH
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, scalar_similarity

NROI=116; CHAR_COLS=["jack","kate","randall","kevin"]
def _nrm(x): return "".join(ch for ch in str(x) if ch.isdigit())
overlap=json.load(open("results/jinrep/03__isrsa_subject_order.json"))
masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}

# neural end-state (mean ISC over 10 runs, masked to 29)
neural=np.load(NEURAL_PATH, allow_pickle=True).item()
neural_end={}
for roi in range(1, NROI+1):
    pg=[]
    for g in [1,2,3]:
        runs=[(np.vstack([np.nanmean(neural[roi,g,r],axis=0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi,g,r]))[:,masks[g]] for r in range(10)]
        pg.append(np.nanmean(np.stack(runs,0),0))
    neural_end[roi-1]=pg

df=pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
df["Character"]=df["Character"].str.lower().str.strip(); df["v"]=df["positive"]-df["negative"]
df=df[df["Run"].between(1,10)]; df["nid"]=df["Participant"].map(_nrm); df["group"]=df["nid"].str[0].astype(int)
piv=df.pivot_table(index=["group","Character","nid"], columns="Run", values="v", aggfunc="mean")
run_cols=np.array(piv.columns, float); vals=piv.values                     # (rows, nruns)

def weighted_dict(gamma):
    w=np.array([gamma**(10.0-r) for r in run_cols])[None,:]                 # gamma=1 flat; gamma->0 recency
    W=np.where(np.isnan(vals), 0.0, w)
    num=np.nansum(np.where(np.isnan(vals),0.0,vals)*W, axis=1); den=W.sum(axis=1)
    ws=np.where(den>0, num/den, np.nan)
    d={}
    for (g,ch,nid),val in zip(piv.index, ws): d.setdefault((g,ch),{})[nid]=val
    return d

def build_sim(dct):
    return [np.array([scalar_similarity([dct.get((g,ch),{}).get(nid,np.nan) for nid in [_nrm(s) for s in overlap[str(g)]]]) for ch in CHAR_COLS]) for g in [1,2,3]]

def run_isrsa(sent_sim):
    R=[np.array([nanspear(neural_end[roi][gi][c_], sent_sim[gi][c_]) for gi in range(3) for c_ in range(sent_sim[gi].shape[0])]) for roi in range(NROI)]
    p=np.array([bootstrap_p(R[roi]) for roi in range(NROI)])
    _,pc,_,_=multipletests(p, alpha=0.01, method="fdr_bh"); posted=np.where(pc<0.01)[0]
    _,pcf,_,_=multipletests(np.minimum(p/2,1.0), alpha=0.05, method="fdr_bh"); figure=np.where(pcf<0.05)[0]
    return posted, figure

print("End-state weighted-combination sweep (w_r = gamma^(10-r)) -- compare to 04c like [24,48,60]:")
for gamma in [1.0, 0.7, 0.5, 0.3, 0.1, 0.0]:
    name={1.0:"flat", 0.0:"recency(R10)"}.get(gamma, f"gamma={gamma}")
    posted, figure = run_isrsa(build_sim(weighted_dict(gamma)))
    print(f"  {name:14s} posted(2s,q<.01): {list(posted)}  | figure-match(1s,q<.05): {list(figure)}")

End-state weighted-combination sweep (w_r = gamma^(10-r)) -- compare to 04c like [24,48,60]:
  flat           posted(2s,q<.01): []  | figure-match(1s,q<.05): []
  gamma=0.7      posted(2s,q<.01): []  | figure-match(1s,q<.05): [np.int64(98), np.int64(100), np.int64(110)]
  gamma=0.5      posted(2s,q<.01): []  | figure-match(1s,q<.05): [np.int64(100)]
  gamma=0.3      posted(2s,q<.01): []  | figure-match(1s,q<.05): []
  gamma=0.1      posted(2s,q<.01): [np.int64(91), np.int64(105)]  | figure-match(1s,q<.05): [np.int64(91), np.int64(105)]
  recency(R10)   posted(2s,q<.01): [np.int64(91), np.int64(105)]  | figure-match(1s,q<.05): [np.int64(91), np.int64(105)]


## 4c.9 · A *psychology-neutral* weight — length / precision weighting (! uses your private transcript)

4c.8's gamma-recency family is principled but bakes in a **cognitive assumption** (recency / impression-updating).
This is a weight that's principled *without* assuming psychology — a **measurement** weight: weight each run's
valence by **how much the subject actually said** about that character that run (token count).

- **Justification is statistical, not cognitive:** more words -> a lower-variance estimate of that run's stance, so
  it should count more (inverse-variance / precision logic). It says nothing about how impressions *form*.
- **It doubles as a no-re-score proxy for concatenation** (4c.10): concatenating a subject's transcript and scoring
  once approximately equals token-weighting the run-level scores.

**Result:** length-weighted end-state -> **0 ROIs** under the posted method (figure-match one-sided: [68, 76, 111,
114]). So the psychology-neutral weight also stays null under the primary threshold — consistent with 4c.6-4c.8:
the sentiment/liking dissociation holds regardless of end-state weighting.

**Public-data status (per the "public-only" rule):** this needs word counts from the **raw transcript** (Jin's
official cleaned transcript, provided by email), which is **not in his public repo** — so this cell is reproducible
only with that local transcript; flag it as such in the write-up. Char-id->name mapping (1=jack,2=kate,3=randall,
4=kevin) was verified against self-mention counts in the transcript.

In [9]:
# 4c.9 · length/precision-weighted end-state IS-RSA  (measurement weight, not a cognitive one)
# NOTE: depends on the RAW TRANSCRIPT, which is NOT in Jin's public repo -> local-only analysis.
import os, json
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests
from config import NEURAL_PATH, TRANSCRIPT_PATH
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, scalar_similarity

NROI=116; CHAR_COLS=["jack","kate","randall","kevin"]
ID2NAME={1:"jack",2:"kate",3:"randall",4:"kevin"}   # verified vs transcript self-mentions
def _nrm(x): return "".join(c for c in str(x) if c.isdigit())
HAVE_TRANSCRIPT = os.path.exists(TRANSCRIPT_PATH)

if not HAVE_TRANSCRIPT:
    print("Raw transcript not found -> skipping (this operationalization is local-only, not public).")
else:
    overlap=json.load(open("results/jinrep/03__isrsa_subject_order.json"))
    masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}

    # neural end-state (mean ISC over 10 runs, masked) -- identical to 4c.6/4c.8
    neural=np.load(NEURAL_PATH, allow_pickle=True).item()
    neural_end={}
    for roi in range(1, NROI+1):
        pg=[]
        for g in [1,2,3]:
            runs=[(np.vstack([np.nanmean(neural[roi,g,r],axis=0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi,g,r]))[:,masks[g]] for r in range(10)]
            pg.append(np.nanmean(np.stack(runs,0),0))
        neural_end[roi-1]=pg

    # per-(subject,run,char) word counts from the transcript
    t=pd.read_csv(TRANSCRIPT_PATH)
    t["nid"]=t["subject"].map(_nrm); t["char"]=t["character"].map(ID2NAME)
    t["wc"]=t["transcribe"].astype(str).str.split().str.len()
    wc=t.groupby(["nid","run","char"],as_index=False)["wc"].sum()

    # run-level valence (public-derived Step-0 output)
    b=pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
    b["char"]=b["Character"].str.lower().str.strip(); b["v"]=b["positive"]-b["negative"]
    b["nid"]=b["Participant"].map(_nrm); b=b[b["Run"].between(1,10)].rename(columns={"Run":"run"})
    m=b.merge(wc,on=["nid","run","char"],how="left"); m["wc"]=m["wc"].fillna(0.0)

    # token-weighted end-state per (nid,char); fall back to flat mean where a subject/char has 0 words
    grp=m.groupby(["nid","char"])
    num=grp.apply(lambda x: np.nansum(x["wc"].values*x["v"].values)); den=grp["wc"].sum()
    flatm=grp["v"].mean()
    ws=(num/den.replace(0,np.nan)).fillna(flatm).reset_index(); ws.columns=["nid","char","v"]
    ws["group"]=ws["nid"].str[0].astype(int)
    lw_dict={}
    for _,r in ws.iterrows(): lw_dict.setdefault((r.group,r.char),{})[r.nid]=r.v

    def build_sim(dct):
        return [np.array([scalar_similarity([dct.get((g,ch),{}).get(nid,np.nan) for nid in [_nrm(s) for s in overlap[str(g)]]]) for ch in CHAR_COLS]) for g in [1,2,3]]
    def run_isrsa(sent_sim, label):
        R=[np.array([nanspear(neural_end[roi][gi][c_], sent_sim[gi][c_]) for gi in range(3) for c_ in range(sent_sim[gi].shape[0])]) for roi in range(NROI)]
        p=np.array([bootstrap_p(R[roi]) for roi in range(NROI)])
        _,pc,_,_=multipletests(p, alpha=0.01, method="fdr_bh"); posted=np.where(pc<0.01)[0]
        _,pcf,_,_=multipletests(np.minimum(p/2,1.0), alpha=0.05, method="fdr_bh"); fig=np.where(pcf<0.05)[0]
        print(f"[{label:26s}] posted(2s,q<.01): {list(posted)}  | figure-match(1s,q<.05): {list(fig)}")

    print("Length/precision-weighted end-state IS-RSA (compare to 04c like [24,48,60]):")
    run_isrsa(build_sim(lw_dict), "length-weighted (neutral)")
    print("\nWeight is measurement-based (amount of evidence), so any regions it finds cannot be dismissed")
    print("as a recency/primacy cognitive assumption. If it matches like where flat did not, that is a")
    print("reliability/precision story; if still ~0, the sentiment/liking dissociation is robust.")


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_8628/436650710.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  num=grp.apply(lambda x: np.nansum(x["wc"].values*x["v"].values)); den=grp["wc"].sum()


Length/precision-weighted end-state IS-RSA (compare to 04c like [24,48,60]):
[length-weighted (neutral) ] posted(2s,q<.01): []  | figure-match(1s,q<.05): [np.int64(68), np.int64(76), np.int64(111), np.int64(114)]

Weight is measurement-based (amount of evidence), so any regions it finds cannot be dismissed
as a recency/primacy cognitive assumption. If it matches like where flat did not, that is a
reliability/precision story; if still ~0, the sentiment/liking dissociation is robust.


## 4c.10 · True concatenation (the holistic measure) — scaffold, requires a Step-0 re-run

Length-weighting (4c.9) approximates concatenation at the *score* level. The **true** holistic measure works at the
*text* level: concatenate each subject's per-character utterances across all runs into one document and score it once.
Because RoBERTa is nonlinear, `score(concat) ≠ mean(run scores)` — it's a genuinely different measurement.

**Recipe (re-run in Step 0, not here):**
1. For each `(subject, character)`, concatenate `transcribe` across runs 1–10 (ordered) → one document.
2. **512-token limit:** most concatenations exceed RoBERTa's max length. Chunk to ≤512 tokens with attention-masked
   **mean-pooling** across chunks (the project's locked pooling convention) — do *not* truncate to the first 512.
3. Score `positive`/`negative` per document → one end-state valence per `(subject, character)`; drop into 4c.6's IS-RSA.

**Caveats to state in the write-up:** (a) chunk-then-pool reintroduces an averaging choice, so this isn't assumption-free
either; (b) longer, more verbose subjects dominate the merged document — that's the point (evidence-weighted) but worth
naming; (c) **transcript-dependent → local-only**, same public-data caveat as 4c.9. **Ran (2026-07-16): 0 ROIs posted (q<0.01), 1 figure-match ROI [55]. Consistent with every other end-state weighting — the holistic concatenation does not recover `like`'s regions [24,48,60] either.**


In [10]:
# 4c.10a · TRUE CONCATENATION — one document per (subject,character), re-scored with RoBERTa.
# RUN LOCALLY (needs cardiffnlp/twitter-roberta-base-sentiment-latest, same model as Step 0). RoBERTa is
# nonlinear, so score(concat) != mean(run scores) -> a genuinely different end-state measure. Chunks to
# <=512 tokens and MEAN-POOLS logits across chunks (project convention), not truncating to the first 512.
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from pathlib import Path

TX = "data/clean_from_jin/socialaha_transcribe_rhea_char_updated_clean_jin.csv"
CHARMAP = {1:"jack", 2:"kate", 3:"randall", 4:"kevin"}       # Jin's coding (confirmed)
df = pd.read_csv(TX); df = df[df["character"].isin(CHARMAP)].copy()
df["char"] = df["character"].map(CHARMAP)
docs = (df.sort_values(["subject","character","run","order"])
          .groupby(["subject","char"])["transcribe"]
          .apply(lambda s: " ".join(str(x) for x in s if pd.notna(x))).reset_index())

_M = "cardiffnlp/twitter-roberta-base-sentiment-latest"
_tok = AutoTokenizer.from_pretrained(_M); _mdl = AutoModelForSequenceClassification.from_pretrained(_M).eval()

@torch.no_grad()
def score_concat(text, maxlen=512):
    ids = _tok(str(text), add_special_tokens=False)["input_ids"]
    if not ids: return np.array([np.nan]*3)
    cls, sep, body = _tok.cls_token_id, _tok.sep_token_id, maxlen-2
    chunks = [ids[i:i+body] for i in range(0, len(ids), body)]
    logits = [_mdl(input_ids=torch.tensor([[cls]+ch+[sep]])).logits.numpy()[0] for ch in chunks]
    return softmax(np.mean(np.vstack(logits), axis=0))       # mean-pool logits -> softmax -> [neg,neu,pos]

rows=[]
for _, r in docs.iterrows():
    neg, neu, pos = score_concat(r["transcribe"])
    ntok = len(_tok(str(r["transcribe"]), add_special_tokens=False)["input_ids"])
    rows.append({"Participant":f"sub-{int(r['subject'])}", "Character":r["char"],
                 "concat_pos":pos, "concat_neg":neg, "concat_neu":neu,
                 "concat_valence":pos-neg, "n_tokens":ntok})
conc = pd.DataFrame(rows)
Path("results/scored").mkdir(parents=True, exist_ok=True)
conc.to_csv("results/scored/00__concat_character_vectors_Twitter_RoB.csv", index=False)
print(f"scored {len(conc)} (subject,character) docs; median tokens = {int(conc.n_tokens.median())}")
print("-> results/scored/00__concat_character_vectors_Twitter_RoB.csv"); print(conc.head())

/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


scored 144 (subject,character) docs; median tokens = 550
-> results/scored/00__concat_character_vectors_Twitter_RoB.csv
  Participant Character  concat_pos  concat_neg  concat_neu  concat_valence  \
0    sub-1001      jack    0.550710    0.050815    0.398475        0.499894   
1    sub-1001      kate    0.140320    0.137721    0.721958        0.002599   
2    sub-1001     kevin    0.365235    0.082155    0.552610        0.283080   
3    sub-1001   randall    0.475132    0.061587    0.463281        0.413544   
4    sub-1005      jack    0.921729    0.015449    0.062822        0.906280   

   n_tokens  
0       532  
1       588  
2       632  
3       595  
4       532  


In [11]:
# 4c.10b · Concatenation end-state IS-RSA — identical pipeline to 4c.6, valence from the concatenated docs (4c.10a).
import os, json
import numpy as np, pandas as pd
from statsmodels.stats.multitest import multipletests
from config import NEURAL_PATH
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, z2r, r2z

NROI=116; CHAR_COLS=["jack","kate","randall","kevin"]
overlap=json.load(open("results/jinrep/03__isrsa_subject_order.json"))
masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}
def _nrm(x): return "".join(ch for ch in str(x) if ch.isdigit())

CONC="results/scored/00__concat_character_vectors_Twitter_RoB.csv"
if not os.path.exists(CONC):
    print("Run 4c.10a first (local RoBERTa scoring) to create", CONC)
else:
    cd=pd.read_csv(CONC); cd["Character"]=cd["Character"].str.lower().str.strip()
    cd["nid"]=cd["Participant"].map(_nrm); cd["group"]=cd["nid"].str[0].astype(int)
    sent_dict={}
    for _,r in cd.iterrows(): sent_dict.setdefault((r.group,r.Character),{})[r.nid]=r.concat_valence
    def scalar_similarity(vals):
        v=np.asarray(vals,float); n=len(v)
        return np.array([-abs(v[i]-v[j]) for i in range(n) for j in range(i+1,n)])
    sent_sim=[]
    for g in [1,2,3]:
        order=[_nrm(s) for s in overlap[str(g)]]
        sent_sim.append(np.array([scalar_similarity([sent_dict.get((g,ch),{}).get(nid,np.nan) for nid in order]) for ch in CHAR_COLS]))
    neural=np.load(NEURAL_PATH, allow_pickle=True).item()
    neural_end={}
    for roi in range(1,NROI+1):
        pg=[]
        for g in [1,2,3]:
            runs=[(np.vstack([np.nanmean(neural[roi,g,r],axis=0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi,g,r]))[:,masks[g]] for r in range(10)]
            pg.append(np.nanmean(np.stack(runs,0),0))
        neural_end[roi-1]=pg
    R=[np.array([nanspear(neural_end[roi][gi][c_], sent_sim[gi][c_]) for gi in range(len(sent_sim)) for c_ in range(sent_sim[gi].shape[0])]) for roi in range(NROI)]
    pvals=[bootstrap_p(R[roi]) for roi in range(NROI)]
    _,pfp,_,_=multipletests(pvals,alpha=0.01,method="fdr_bh"); sig_posted=np.where(pfp<0.01)[0]
    _,pff,_,_=multipletests(np.minimum(np.array(pvals)/2,1.0),alpha=0.05,method="fdr_bh"); sig_fig=np.where(pff<0.05)[0]
    mean_r=np.array([z2r(np.nanmean(r2z(R[roi]))) for roi in range(NROI)])
    print("Concatenation end-state sentiment IS-RSA:")
    print(f"  Posted (2s, q<.01):     {len(sig_posted)} ROIs {list(sig_posted)}")
    print(f"  Figure-Match (1s, q<.05): {len(sig_fig)} ROIs {list(sig_fig)}")
    print("  compare -> flat end-state (4c.6)=0 ; like [24,48,60] ; length-weighted (4c.9) posted []")
    np.save("results/IS-RSA/04c__concat_end_state_isrsa.npy",{"mean_r":mean_r,"p":pvals,"p_fdr":pff,"sig_rois":sig_fig},allow_pickle=True)

Concatenation end-state sentiment IS-RSA:
  Posted (2s, q<.01):     0 ROIs []
  Figure-Match (1s, q<.05): 1 ROIs [np.int64(55)]
  compare -> flat end-state (4c.6)=0 ; like [24,48,60] ; length-weighted (4c.9) posted []


## 4c.11 · Choosing among operationalizations without circularity (specification-curve summary)

There is **no single "true"** end-state operationalization — each is a different measurement model of a latent
construct. "Best" is a trap: if "best" quietly means "most significant ROIs," that's the double-dipping this project
already warns against. Two legitimate strategies, both used above:

**Pre-specify a primary by a brain-blind criterion.** Legal criteria: construct-match to the research question, and
**reliability** (computed on behavior alone). Illegal: number of significant ROIs. -> Primary = **flat end-state**
(4c.6, the most reliable public-data valence measure); everything else is exploratory.

**Report the whole set of weightings, not a winner** (4c.7-4c.10): show the result under every defensible choice and
ask whether the neural coupling is *robust* to the weighting, rather than cherry-picking. Here it is robust —
sentiment stays null in the `like` regions under flat, recency, gamma-family, length weighting, AND true concatenation;
only strongly recency-weighted settings surface a separate [91,105] signal.

| Operationalization | Assumption type | Public-only? | Role | Result (posted q<.01) |
|---|---|---|---|---|
| Flat average (4c.6) | weak (equal weight) | yes (derived valence) | **primary** | 0 ROIs |
| Recency / Run-10 (4c.7) | cognitive (recency) | yes | exploratory extreme | [91,105] |
| gamma-recency family (4c.8) | cognitive (recency), a-priori | yes | exploratory curve | [] until gamma<=0.1 -> [91,105] |
| Length / precision (4c.9) | **measurement** (evidence) | raw transcript = local-only | neutral alternative | 0 (fig-match [68,76,111,114]) |
| True concatenation (4c.10) | measurement + chunk-pool | local-only | holistic follow-up | 0 (fig-match [55]) |

**The one rule that keeps all of this honest:** the choice among operationalizations is fixed by criteria that never
look at the brain–behavior outcome. Reliability qualifies; fit to the neural data does not.

## 4c.12 · Positive-valence subscale — the aggregate alternative to `like` vs `positive_emotion`

Instead of choosing one item, average the liking/warmth/affiliation items into one composite. **Result:** reliability **0.151** (beats PC1 0.055 and positive_emotion 0.035, below `like` 0.196) and detects **[65] RH DorsalAttn (SPL)** — one region, *different* from `like`'s [24,48,60]. So a positive-valence aggregate is a stronger measure than either single item or PC1, but it localizes elsewhere and stays below `like`'s reliability — reinforcing that `like` (viewer stance) is the special, reliably-measured signal. Item list is editable at the top.

In [12]:
# 4c.12 · Positive-valence SUBSCALE (aggregate alternative to picking `like` vs `positive_emotion`).
# Mean of z-scored liking/warmth/affiliation items -> a more reliable aggregate than PC1 or the single
# positive_emotion item. Same IS-RSA as 4c.2 (scalar AnnaK similarity, mean-ISC end-state, bootstrap, FDR).
# ADDED ALONGSIDE PC1/positive_emotion/like -- does not replace them.
import numpy as np, pandas as pd, json as _json, scipy.io as sio
from pathlib import Path
from itertools import combinations
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from helpers import _pair_mask, rearrange_new, nanspear, bootstrap_p, z2r, r2z, NROI
from config import SURVEY_DIR, NEURAL_PATH
CHAR_COLS=["jack","kate","randall","kevin"]
BLK={"data1":["warm and kind","intelligent","agreeable","extraverted","impulsive","emotionally stable","open-minded","trustworthy","competent"],
 "data2":["rational behavior","positive emotion","good relationship"],"data3":["empathize","understand","like","similar"],
 "data4":["friendly","assertive","reserved","self-disciplined","optimistic","humorous","sincere and honest","self-centered","determined","caring and supportive","ambitious"],
 "data5":["good job","content","big influence"],"data6":["want to be character","want to be friends","care what happens","annoying","attractive"]}
ALL=[l for b in ["data1","data2","data3","data4","data5","data6"] for l in BLK[b]]
# EDITABLE positive-valence subscale (liking/warmth/affiliation; excludes ability, neutral, negative):
POSVAL=["warm and kind","trustworthy","agreeable","positive emotion","good relationship","empathize",
        "understand","like","friendly","caring and supportive","want to be friends","want to be character","attractive"]
def _nrm(x): return "".join(c for c in str(x) if c.isdigit())
rows=[]
for f in sorted(p for p in Path(SURVEY_DIR).glob("*.mat") if p.name!="labels.mat"):
    m=sio.loadmat(f)
    if not all(f"data{i}" in m for i in range(1,7)): continue
    mat=np.vstack([np.asarray(m[b],float) for b in ["data1","data2","data3","data4","data5","data6"]])
    for ci,ch in enumerate(CHAR_COLS):
        rec={"nid":_nrm(Path(f).stem),"group":int(_nrm(Path(f).stem)[0]),"Character":ch}
        for ri,lab in enumerate(ALL): rec[lab]=mat[ri,ci]
        rows.append(rec)
sv=pd.DataFrame(rows)
Z=(sv[POSVAL]-sv[POSVAL].mean())/sv[POSVAL].std(ddof=0); sv["posvalence"]=Z.mean(axis=1)
sub={}
for _,r in sv.iterrows(): sub.setdefault((r.group,r.Character),{})[r.nid]=r.posvalence
overlap=_json.load(open("results/jinrep/03__isrsa_subject_order.json"))
def _ssim(v): v=np.asarray(v,float); n=len(v); return np.array([-abs(v[i]-v[j]) for i in range(n) for j in range(i+1,n)])
rels=[]
for grp in [1,2,3]:
    order=[_nrm(s) for s in overlap[str(grp)]]; sim={ch:_ssim([sub.get((grp,ch),{}).get(nid,np.nan) for nid in order]) for ch in CHAR_COLS}
    for a in combinations(range(4),2):
        b=tuple(i for i in range(4) if i not in a)
        A=np.nanmean([sim[CHAR_COLS[i]] for i in a],0); B=np.nanmean([sim[CHAR_COLS[i]] for i in b],0); mm=~(np.isnan(A)|np.isnan(B))
        if mm.sum()>5: rels.append(spearmanr(A[mm],B[mm])[0])
reliab=float(np.nanmean(rels))
masks={g:_pair_mask(g, overlap[str(g)]) for g in [1,2,3]}
neural=np.load(NEURAL_PATH,allow_pickle=True).item(); neural_end={}
for roi in range(1,NROI+1):
    pg=[]
    for g in [1,2,3]:
        runs=[(np.vstack([np.nanmean(neural[roi,g,r],0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi,g,r]))[:,masks[g]] for r in range(10)]
        pg.append(np.nanmean(np.stack(runs,0),0))
    neural_end[roi-1]=pg
ssim=[np.array([_ssim([sub.get((g,ch),{}).get(nid,np.nan) for nid in [_nrm(s) for s in overlap[str(g)]]]) for ch in CHAR_COLS]) for g in [1,2,3]]
R=[np.array([nanspear(neural_end[roi][gi][c_], ssim[gi][c_]) for gi in range(3) for c_ in range(ssim[gi].shape[0])]) for roi in range(NROI)]
p=np.array([bootstrap_p(R[roi]) for roi in range(NROI)])
_,pc,_,_=multipletests(p,alpha=0.01,method="fdr_bh"); posted=np.where(pc<0.01)[0]
_,pcf,_,_=multipletests(np.minimum(p/2,1.0),alpha=0.05,method="fdr_bh"); figs=np.where(pcf<0.05)[0]
mean_r=np.array([z2r(np.nanmean(r2z(R[roi]))) for roi in range(NROI)])
np.save("results/IS-RSA/04c__survey_posvalence_isrsa.npy",{"mean_r":mean_r,"p":p,"p_fdr":pc,"sig_rois":posted,"reliability":reliab,"items":POSVAL},allow_pickle=True)
print(f"positive-valence subscale = mean of {len(POSVAL)} z-scored items")
print(f"cross-char reliability = {reliab:.3f}   (vs like 0.196, PC1 0.055, positive_emotion 0.035)")
print(f"posted (2s,q<.01): {[int(x) for x in posted]}  | figure-match (1s,q<.05): {[int(x) for x in figs]}")

/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_8628/3453606240.py:42: RuntimeWarning: Mean of empty slice
  A=np.nanmean([sim[CHAR_COLS[i]] for i in a],0); B=np.nanmean([sim[CHAR_COLS[i]] for i in b],0); mm=~(np.isnan(A)|np.isnan(B))


positive-valence subscale = mean of 13 z-scored items
cross-char reliability = 0.151   (vs like 0.196, PC1 0.055, positive_emotion 0.035)
posted (2s,q<.01): [65]  | figure-match (1s,q<.05): [65]
